In [ ]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
import evaluate
import nltk
import matplotlib.pyplot as plt
nltk.download("wordnet", quiet=True)

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed
)

# ======================
# CONFIG (UPDATED WITH DYNAMIC LR)
# ======================
MODEL_NAME = "Osame1/AraT5-msa-small-finetuned-xlsum-ar"
DATA_FILE  = "/kaggle/input/versionfinal2/merged_datasetfinalv2 (2).csv"
OUTPUT_DIR = "/kaggle/working/t5-arabic-advenced_finetuning"

MAX_INPUT_LENGTH  = 384
MAX_TARGET_LENGTH = 96
BATCH_SIZE = 4
EPOCHS = 3
LR = 3e-4  # ☢ CRITICAL: Reduced from 1e-3 for stable fine-tuning
SEED = 42
BETA_SALIENCE = 0.3
WARMUP_RATIO = 0.1  # 10% of training steps for warmup phase

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ======================
# LOAD DATA
# ======================
def load_csv(path):
    # Add a check for file existence
    if not os.path.exists(path):
        raise FileNotFoundError(f"Error: The data file '{path}' was not found. Please ensure the cell that generates this file has been run successfully and Google Drive is mounted.")

    df = pd.read_csv(path)
    data = {"text": [], "summary": []}
    for _, r in tqdm(df.iterrows(), total=len(df), desc="Loading data"):
        t, s = str(r["text"]).strip(), str(r["summary"]).strip()
        if t and s:
            data["text"].append(t)
            data["summary"].append(s)
    return Dataset.from_dict(data)

dataset = load_csv(DATA_FILE)

# ======================
# TOKENIZER
# ======================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def prompt(x):
    return [f"\u0627\u0644\u0646\u0635: {t}\n\u0627\u0644\u0645\u0644\u062e\u0635:" for t in x]

# ======================
# WEAK SALIENCE LABELS
# ======================
def build_salience(input_ids, summary_ids):
    summary_set = set([i for i in summary_ids if i != tokenizer.pad_token_id])
    return [1 if tok in summary_set else 0 for tok in input_ids]

# ======================
# PREPROCESS
# ======================
def preprocess(examples):
    inputs  = prompt(examples["text"])
    targets = examples["summary"]

    enc = tokenizer(inputs, max_length=MAX_INPUT_LENGTH,
                    truncation=True, padding="max_length")

    # Fix: Replace as_target_tokenizer with text_target
    dec = tokenizer(text_target=targets, max_length=MAX_TARGET_LENGTH,
                    truncation=True, padding="max_length")

    labels, salience = [], []

    for inp, lab in zip(enc["input_ids"], dec["input_ids"]):
        lab_masked = [(t if t != tokenizer.pad_token_id else -100) for t in lab]
        labels.append(lab_masked)
        salience.append(build_salience(inp, lab))

    enc["labels"] = labels
    enc["salience_labels"] = salience
    return enc

dataset_tok = dataset.map(preprocess, batched=True,
                          remove_columns=dataset.column_names)

# train/validation split
dataset_tok = dataset_tok.train_test_split(test_size=0.1, seed=SEED)
train_ds = dataset_tok["train"]
eval_ds  = dataset_tok["test"]

# ======================
# HIERARCHICAL T5 MODEL
# ======================
class HierarchicalT5(nn.Module):
    def __init__(self, name, adapter_dim=256):
        super().__init__()
        self.base = AutoModelForSeq2SeqLM.from_pretrained(name)
        d = self.base.config.d_model

        self.adapter = nn.Sequential(
            nn.Linear(d, adapter_dim),
            nn.ReLU(),
            nn.Linear(adapter_dim, d)
        )

        self.sent_proj = nn.Linear(d, d)
        self.salience_head = nn.Linear(d, 1)

        # Configure generation parameters
        self.base.generation_config.num_beams = 4
        self.base.generation_config.min_length = 10
        self.base.generation_config.max_length = MAX_TARGET_LENGTH

    @property
    def config(self):
        """Expose base model's config to satisfy Trainer requirements"""
        return self.base.config

    @property
    def device(self):
        """Expose device for compatibility"""
        return next(self.parameters()).device

    @property
    def generation_config(self):
        return self.base.generation_config

    def sentence_context(self, hidden, attention_mask):
        B, T, D = hidden.shape
        chunk = 20
        ctx = torch.zeros_like(hidden)

        for i in range(0, T, chunk):
            h = hidden[:, i:i+chunk, :]
            m = attention_mask[:, i:i+chunk].unsqueeze(-1)
            pooled = (h * m).sum(1) / (m.sum(1) + 1e-8)
            pooled = self.sent_proj(pooled).unsqueeze(1)
            ctx[:, i:i+chunk, :] = pooled

        return ctx

    def forward(self, input_ids=None, attention_mask=None,
                labels=None, salience_labels=None, **kwargs):

        enc_out = self.base.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        H = enc_out.last_hidden_state
        H = H + self.sentence_context(H, attention_mask)
        H = H + self.adapter(H)

        salience_logits = self.salience_head(H).squeeze(-1)

        outputs = self.base(
            input_ids=None,
            attention_mask=attention_mask,
            encoder_outputs=(H,),
            labels=labels,
            return_dict=True
        )

        loss = outputs.loss

        if salience_labels is not None:
            salience_labels = salience_labels.float()
            sal_loss = F.binary_cross_entropy_with_logits(
                salience_logits, salience_labels)
            loss = loss + BETA_SALIENCE * sal_loss

        outputs.loss = loss
        return outputs

    def generate(self, *args, **kwargs):
        kwargs.pop('salience_labels', None)
        kwargs["use_cache"] = False
        return self.base.generate(*args, **kwargs)

model = HierarchicalT5(MODEL_NAME)

# ======================
# DATA COLLATOR
# ======================
class CustomCollator(DataCollatorForSeq2Seq):
    def __call__(self, features):
        sal = torch.tensor([f.pop("salience_labels") for f in features])
        batch = super().__call__(features)
        batch["salience_labels"] = sal
        return batch

collator = CustomCollator(tokenizer, model=model.base, label_pad_token_id=-100)

# ======================
# METRICS
# ======================
# Ensure evaluate, bert_score, sacrebleu are installed globally in the cell
# Removed try-except blocks as !pip install at the top should cover it.

bertscore_metric = evaluate.load("bertscore")
chrf_metric      = evaluate.load("chrf")
bleu_metric      = evaluate.load("bleu")
meteor_metric    = evaluate.load("meteor")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    # \u062a\u062d\u0648\u064a\u0644 \u0627\u0644\u062a\u0646\u0633\u0648\u0631\u0627\u062a \u0625\u0644\u0649 NumPy \u0625\u0630\u0627 \u0644\u0632\u0645 \u0627\u0644\u0623\u0645\u0631
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    vocab_size = tokenizer.vocab_size

    # ====== \u0625\u0635\u0644\u0627\u062d \u062d\u0627\u0633\u0645: \u062a\u0646\u0642\u064a\u0629 \u0627\u0644\u0631\u0645\u0648\u0632 \u063a\u064a\u0631 \u0627\u0644\u0635\u0627\u0644\u062d\u0629 ======
    # 1. \u062a\u062d\u0648\u064a\u0644 \u0625\u0644\u0649 \u0623\u0639\u062f\u0627\u062f \u0635\u062d\u064a\u062d\u0629
    preds = preds.astype(np.int64)

    # 2. \u0627\u0633\u062a\u0628\u062f\u0627\u0644 \u0627\u0644\u0642\u064a\u0645 \u0627\u0644\u0633\u0627\u0644\u0628\u0629 \u0628\u0640 pad_token
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)

    # 3. \u0627\u0642\u062a\u0637\u0627\u0639 \u0627\u0644\u0642\u064a\u0645 \u0627\u0644\u0623\u0643\u0628\u0631 \u0645\u0646 \u062d\u062c\u0645 \u0627\u0644\u0642\u0627\u0645\u0648\u0633
    preds = np.clip(preds, 0, vocab_size - 1)

    # \u0645\u0639\u0627\u0644\u062c\u0629 \u0627\u0644\u062a\u0633\u0645\u064a\u0627\u062a (labels)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.clip(labels.astype(np.int64), 0, vocab_size - 1)
    # ==============================================

    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # BERTScore
    bertscore = bertscore_metric.compute(predictions=decoded_preds,
                                         references=decoded_labels,
                                         lang="ar")

    # chrF
    chrf = chrf_metric.compute(predictions=decoded_preds,
                               references=decoded_labels)

    # BLEU
    bleu = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels]
    )

    # METEOR
    meteor = meteor_metric.compute(predictions=decoded_preds,
                                   references=decoded_labels)

    return {
        "bertscore_precision": np.mean(bertscore["precision"]),
        "bertscore_recall":    np.mean(bertscore["recall"]),
        "bertscore_f1":        np.mean(bertscore["f1"]),
        "chrf": chrf["score"],
        "bleu": bleu["bleu"],
        "meteor": meteor["meteor"],
    }

# ======================
# TRAINING ARGS (WITH DYNAMIC LR SCHEDULER)
# ======================
args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    # 🔑 KEY CHANGES FOR CLEAN PROGRESS:
    logging_strategy="epoch",      # Only log at epoch end (no step-by-step logs)
    logging_steps=10,              # Ignored when logging_strategy="epoch"
    disable_tqdm=False,            # Ensure tqdm progress bar is visible (single-line)
    log_level="error",             # Suppress non-critical logs
    log_level_replica="error",
    predict_with_generate=True,
    fp16=False,
    report_to="none",
    seed=SEED,
    logging_dir=f"{OUTPUT_DIR}/logs",
    save_safetensors=False
)
# ======================
# TRAINER
# ======================
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    # Re-applying fix: 'tokenizer' argument is deprecated in Seq2SeqTrainer
    # tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)
from transformers import TrainerCallback
import time
import numpy as np

class EpochSummaryCallback(TrainerCallback):
    """Prints a clean epoch-level summary with metrics."""
    def __init__(self):
        self.start_time = None

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f"\n[{state.global_step}/{state.max_steps} {time.strftime('%M:%S', time.gmtime(0))}, Epoch {int(state.epoch)+1}/{int(args.num_train_epochs)}]")
        print(f"{'Epoch':<6} {'Training Loss':<14} {'Validation Loss':<16} {'BERTScore F1':<14} {'chrF':<6} {'BLEU':<6} {'METEOR':<7}")

    def on_epoch_end(self, args, state, control, **kwargs):
        # Get last logged losses and metrics
        train_loss = None
        eval_loss = None
        bert_f1 = 0
        chrf_score = 0
        bleu_score = 0
        meteor_score = 0

        # Iterate log history to find last relevant logs for this epoch
        for log in reversed(state.log_history):
            if "epoch" in log and int(log["epoch"]) == int(state.epoch):
                train_loss = log.get("loss", train_loss)
                eval_loss = log.get("eval_loss", eval_loss)
                bert_f1 = log.get("eval_bertscore_f1", bert_f1)
                chrf_score = log.get("eval_chrf", chrf_score)
                bleu_score = log.get("eval_bleu", bleu_score)
                meteor_score = log.get("eval_meteor", meteor_score)
                break

        print(f"{int(state.epoch)+1:<6} "
              f"{(train_loss if train_loss is not None else 0):<14.4f} "
              f"{(eval_loss if eval_loss is not None else 0):<16.4f} "
              f"{bert_f1:<14.4f} "
              f"{chrf_score:<6.2f} "
              f"{bleu_score:<6.2f} "
              f"{meteor_score:<7.2f}")

print(f" Training with DYNAMIC LR (Cosine + {int(WARMUP_RATIO*100)}% warmup)...")
print(f"   Initial LR: {LR}, Warmup steps: ~{int(len(train_ds) * WARMUP_RATIO / BATCH_SIZE)}")
trainer.train()

# ======================
# FINAL METRICS + PERPLEXITY
# ======================
metrics = trainer.evaluate()
if "eval_loss" in metrics and metrics["eval_loss"] is not None:
    metrics["perplexity"] = math.exp(metrics["eval_loss"])

print("\n" + "="*50)
print("FINAL EVALUATION METRICS")
print("="*50)
for k, v in sorted(metrics.items()):
    if isinstance(v, float):
        print(f"{k:30s}: {v:.4f}")
    else:
        print(f"{k:30s}: {v}")
print("="*50)

# ======================
# SAVE MODEL
# ======================
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Model saved to: {OUTPUT_DIR}")

# ======================
# PLOTS (INCLUDING LEARNING RATE SCHEDULE)
# ======================
log_history = trainer.state.log_history

# Extract metrics
train_loss, eval_loss = [], []
bertscore_f1, chrf_scores, bleu_scores = [], [], []
learning_rates = []  # Track LR evolution

for log in log_history:
    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
    if "eval_loss" in log:
        eval_loss.append(log["eval_loss"])
        bertscore_f1.append(log.get("eval_bertscore_f1", 0))
        chrf_scores.append(log.get("eval_chrf", 0))
        bleu_scores.append(log.get("eval_bleu", 0))
    if "learning_rate" in log:
        learning_rates.append(log["learning_rate"])

epochs_range = range(1, len(eval_loss) + 1)

# Create plots directory
plots_dir = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(plots_dir, exist_ok=True)

# Plot 1: Training Loss
plt.figure(figsize=(8, 4.5))
plt.plot(train_loss, linewidth=2)
plt.title("Training Loss", fontsize=14, fontweight='bold')
plt.xlabel("Training Step", fontsize=11)
plt.ylabel("Loss", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "train_loss.png"), dpi=150)
plt.close()

# Plot 2: Validation Loss
plt.figure(figsize=(8, 4.5))
plt.plot(epochs_range, eval_loss, marker='o', linewidth=2, markersize=6, color='#2E86AB')
plt.title("Validation Loss", fontsize=14, fontweight='bold')
plt.xlabel("Epoch", fontsize=11)
plt.ylabel("Loss", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "val_loss.png"), dpi=150)
plt.close()

# Plot 3: Learning Rate Schedule (NEW)
plt.figure(figsize=(8, 4.5))
plt.plot(learning_rates, color='#A23B72', linewidth=2.5)
plt.title(f"Learning Rate Schedule (Cosine + {int(WARMUP_RATIO*100)}% Warmup)",
          fontsize=14, fontweight='bold')
plt.xlabel("Training Step (logged every 10 steps)", fontsize=11)
plt.ylabel("Learning Rate", fontsize=11)
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "lr_schedule.png"), dpi=150)
plt.close()

# Plot 4: BERTScore F1
plt.figure(figsize=(8, 4.5))
plt.plot(epochs_range, bertscore_f1, marker='o', linewidth=2, markersize=6, color='#2E86AB')
plt.title("BERTScore F1", fontsize=14, fontweight='bold')
plt.xlabel("Epoch", fontsize=11)
plt.ylabel("F1 Score", fontsize=11)
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "bertscore_f1.png"), dpi=150)
plt.close()

# Plot 5: chrF
plt.figure(figsize=(8, 4.5))
plt.plot(epochs_range, chrf_scores, marker='o', linewidth=2, markersize=6, color='#F18F01')
plt.title("chrF Score", fontsize=14, fontweight='bold')
plt.xlabel("Epoch", fontsize=11)
plt.ylabel("Score", fontsize=11)
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "chrf.png"), dpi=150)
plt.close()

# Plot 6: BLEU
plt.figure(figsize=(8, 4.5))
plt.plot(epochs_range, bleu_scores, marker='o', linewidth=2, markersize=6, color='#C73E1D')
plt.title("BLEU Score", fontsize=14, fontweight='bold')
plt.xlabel("Epoch", fontsize=11)
plt.ylabel("Score", fontsize=11)
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "bleu.png"), dpi=150)
plt.close()

print(f"\n✅ All plots saved to: {plots_dir}") 